# OSC Injection-Extraction Accuracy — v3.1 (Hungarian, region-canon, range-aware)

Scores OSC distillation (`injection-structured.json`) against the **human-corrected**
ground truth, using **global-optimal (Hungarian) assignment** so multi-site papers
are paired to minimise *total* error — no greedy cross-pairing artifacts.

**v3.1 vs v3:**
- **Hungarian matching** (`scipy.optimize.linear_sum_assignment`) per (PMID, region)
  group — the rigorous fix for multi-site papers. Proven: it does NOT inflate the
  score; it reveals genuine residuals instead of hiding them behind bad pairing.
- **Region-canon join** (`canon()`) — fixes the v2 bug where 53/95 rows silently
  fell into "no region" because GT `FIELD CA1`/`NUCLEUS ACCUMBENS` didn't match the
  `REGION_KEYS` substrings. All rows now score.
- **Range-aware magnitude** — bilateral `±X` and range `"-1.75 to -2.75"` matched on
  the closest endpoint magnitude (Rule A/B aware).
- **Injection-only** — `cannula_guide` excluded via `target_type` or `KNOWN_CANNULA`.

Set the two paths in Cell 1, run top to bottom.

In [5]:
# Cell 1 — config
import csv, json, re
from collections import defaultdict
import numpy as np
from scipy.optimize import linear_sum_assignment

GT_CSV   = 'verification_sheet.csv'      # human-corrected ground truth (needs region + corrected ap/ml/dv)
OSC_JSON = r'C:\Users\asathyanesan1\Documents\GitHub\Neuroinjector\react-app\public\data\injection-structured.json'   # OSC distillation output (14k)
TOL      = 0.25                          # mm

# PMIDs that are cannula/guide placements, not injections. Bridges current data
# (no target_type yet). Empty once a Rule-C re-run tags them 'cannula_guide'.
KNOWN_CANNULA = {'27924869'}

# canonical region key -> substrings that may appear in an OSC ccf_region
REGION_KEYS = {
    'CA1': ['ca1', 'field ca1', 'hippocamp'],
    'CP':  ['caudoputamen', 'cp', 'striat', 'dorsal striatum',
            'dorsolateral striatum', 'dorsomedial striatum', 'ds'],
    'ACB': ['accumbens', 'acb', 'nac', 'nucleus accumbens'],
}
print('config | tol', TOL, 'mm | known cannula', KNOWN_CANNULA)

config | tol 0.25 mm | known cannula {'27924869'}


In [6]:
# Cell 2 — helpers: canon region + range-aware magnitude matching
def canon(region):
    """Map any region label (GT full-name OR OSC ccf_region) to CA1/CP/ACB."""
    r = (region or '').lower()
    if 'ca1' in r or 'hippocamp' in r: return 'CA1'
    if 'accumbens' in r or 'acb' in r or 'nac' in r: return 'ACB'
    if 'striat' in r or 'caudoput' in r or r.strip() in ('cp','ds'): return 'CP'
    return (region or '').strip().upper()

def to_num(v):
    if v is None: return None
    if isinstance(v, (int, float)): return float(v)
    s = str(v).replace('\u2212','-').strip()
    if not s or s.lower() in ('none','null','na','n/a',''): return None
    nums = re.findall(r'-?\d+\.?\d*', s)
    return sum(float(n) for n in nums)/len(nums) if nums else None

def endpoints(v):
    if v is None: return []
    if isinstance(v, (int, float)): return [float(v)]
    return [float(n) for n in re.findall(r'-?\d+\.?\d*', str(v).replace('\u2212','-'))] or []

def best_abs_err(gt_val, osc_raw, ap=False):
    """min |error| over OSC endpoints. ml/dv on magnitude; ap sign-tolerant
    (some papers omit the minus). This is the range-aware Rule-A/B fix."""
    if gt_val is None: return None
    eps = endpoints(osc_raw)
    if not eps: return None
    if ap:
        return min(min(abs(gt_val-e), abs(abs(gt_val)-abs(e))) for e in eps)
    g = abs(gt_val)
    return min(abs(g-abs(e)) for e in eps)
print('helpers ready')

helpers ready


In [7]:
# Cell 3 — load ground truth (tolerant columns; canon region; template guard)
gt = defaultdict(list)
with open(GT_CSV, encoding='utf-8-sig') as f:
    reader = csv.DictReader(f); cols = reader.fieldnames or []
    # prefer a clean 'region' col; else derive from ccf_region via canon()
    reg_col = 'region' if 'region' in cols else ('ccf_region' if 'ccf_region' in cols else None)
    if reg_col is None: raise KeyError(f'no region column in {cols}')
    rows = list(reader)

filled = sum(1 for r in rows if (r.get('reviewer_confirms') or '').strip())
if filled == 0 and 'orig_ml_mm' not in cols:
    print('#'*70)
    print('WARNING: reviewer_confirms empty & no orig_ml_mm -> looks like the')
    print('UNLABELED template. Its ap/ml/dv may be pipeline-derived, not human-')
    print('corrected. Point GT_CSV at the LABELED sheet for a valid number.')
    print('#'*70)

for r in rows:
    pmid   = str(r['pmid']).strip()
    region = canon(r[reg_col])
    ap, ml, dv = to_num(r['ap_mm']), to_num(r['ml_mm']), to_num(r['dv_mm'])
    if ap is None and ml is None and dv is None: continue
    gt[(pmid, region)].append({'ap':ap,
        'ml': None if ml is None else abs(ml),
        'dv': None if dv is None else abs(dv),
        'dv_signed': dv, 'quote': (r.get('source_quote') or '')[:120]})

multi = sum(1 for v in gt.values() if len(v) > 1)
print(f'GT: {sum(len(v) for v in gt.values())} rows | {len(gt)} (pmid,region) groups '
      f'| {len({p for p,_ in gt})} PMIDs | {filled} labeled | {multi} multi-site groups')

GT: 95 rows | 70 (pmid,region) groups | 65 PMIDs | 95 labeled | 22 multi-site groups


In [8]:
# Cell 4 — load OSC + injection-only filter (region via canon or substring)
osc_raw = json.load(open(OSC_JSON))
osc = osc_raw if isinstance(osc_raw, dict) else {str(x.get('pmid')): x for x in osc_raw}

def is_injection(pmid, t):
    if not isinstance(t, dict): return False
    if (t.get('target_type') or '').lower() == 'cannula_guide': return False
    if str(pmid) in KNOWN_CANNULA: return False
    return True

def osc_targets(pmid, region):
    rec = osc.get(str(pmid))
    if not isinstance(rec, dict): return []
    keys = REGION_KEYS.get(region, [region.lower()])
    out = []
    for t in rec.get('targets', []):
        if not is_injection(pmid, t): continue
        reg = (t.get('ccf_region') or '')
        if any(k in reg.lower() for k in keys) or canon(reg) == region:
            out.append({'ap':to_num(t.get('ap_mm')),'ml':to_num(t.get('ml_mm')),
                        'dv':to_num(t.get('dv_mm')),'ap_raw':t.get('ap_mm'),
                        'ml_raw':t.get('ml_mm'),'dv_raw':t.get('dv_mm')})
    return out

n_inj = n_can = 0
for pmid, rec in osc.items():
    if not isinstance(rec, dict): continue
    for t in rec.get('targets', []):
        if not isinstance(t, dict): continue
        n_inj += is_injection(pmid, t); n_can += (not is_injection(pmid, t))
print(f'OSC targets — injection: {n_inj} | cannula/excluded: {n_can} | papers: {len(osc)}')

OSC targets — injection: 13044 | cannula/excluded: 4 | papers: 14057


In [9]:
# Cell 5 — HUNGARIAN assignment per (pmid, region) group
def pair_cost(g, c):
    errs = [best_abs_err(g[k], c[k+'_raw'], ap=(k=='ap'))
            for k in ('ap','ml','dv')]
    errs = [e for e in errs if e is not None]
    return sum(errs)/len(errs) if errs else 9e9

def row_ok(g, c):
    for k in ('ap','ml','dv'):
        if g[k] is None: continue
        e = best_abs_err(g[k], c[k+'_raw'], ap=(k=='ap'))
        if e is None or e > TOL: return False
    return True

results, missing, no_region = [], [], []
for (pmid, region), grows in gt.items():
    if str(pmid) not in osc:
        missing.append((pmid, region)); continue
    cand = [c for c in osc_targets(pmid, region)
            if any(c[k] is not None for k in ('ap','ml','dv'))]
    if not cand:
        no_region.append((pmid, region)); continue
    if len(grows) == 1 or len(cand) == 1:               # trivial: nearest
        used = set()
        for g in grows:
            costs = [(pair_cost(g, c), i) for i, c in enumerate(cand) if i not in used]
            _, bi = min(costs) if costs else (0, 0)
            used.add(bi); results.append({'pmid':pmid,'region':region,'gt':g,'osc':cand[bi]})
    else:                                                # global-optimal
        C = np.array([[pair_cost(g, c) for c in cand] for g in grows])
        ri, ci = linear_sum_assignment(C)
        for r_, c_ in zip(ri, ci):
            results.append({'pmid':pmid,'region':region,'gt':grows[r_],'osc':cand[c_]})

print(f'matched {len(results)} GT rows | missing PMID {len(missing)} | no region {len(no_region)}')

matched 84 GT rows | missing PMID 3 | no region 4


In [10]:
# Cell 6 — per-axis + full 3-axis accuracy (range-aware, Hungarian-paired)
print(f'{"axis":5} {"n":>4} {"match<=.25":>12} {"MAE":>8} {"median":>8}')
for axis, lab in (('ap','AP'), ('ml','|ML|'), ('dv','|DV|')):
    errs = []
    for r in results:
        if r['gt'][axis] is None: continue
        e = best_abs_err(r['gt'][axis], r['osc'][axis+'_raw'], ap=(axis=='ap'))
        if e is not None: errs.append(e)
    n = len(errs); ok = sum(e <= TOL for e in errs)
    mae = sum(errs)/n if n else 0; med = sorted(errs)[n//2] if n else 0
    print(f'{lab:5} {n:>4} {ok:>4}/{n:<3}({100*ok//max(n,1):>3}%) {mae:>8.3f} {med:>8.3f}')

full = [r for r in results if all(r['gt'][k] is not None for k in ('ap','ml','dv'))]
ok3 = sum(row_ok(r['gt'], r['osc']) for r in full)
print(f'\nFULL 3-axis within +/-{TOL}mm (Hungarian): {ok3}/{len(full)} '
      f'({100*ok3//max(len(full),1)}%)   <-- real accuracy')

axis     n   match<=.25      MAE   median
AP      84   83/84 ( 98%)    0.006    0.000
|ML|    84   84/84 (100%)    0.000    0.000
|DV|    84   83/84 ( 98%)    0.015    0.000

FULL 3-axis within +/-0.25mm (Hungarian): 82/84 (97%)   <-- real accuracy


In [11]:
# Cell 7 — per-region + DV sign convention
per = defaultdict(lambda: [0,0])
for r in full:
    per[r['region']][1] += 1
    if row_ok(r['gt'], r['osc']): per[r['region']][0] += 1
print('region  corrected 3-axis')
for k in sorted(per):
    o, n = per[k]; print(f'  {k:4} {o:>3}/{n:<3} ({100*o//max(n,1)}%)')

sp = [(r['gt']['dv_signed'], r['osc']['dv_raw']) for r in results
      if r['gt']['dv_signed'] is not None and to_num(r['osc']['dv_raw']) is not None]
if sp:
    agree = sum((a<0)==(to_num(b)<0) for a,b in sp)
    print(f'\nDV sign agreement: {agree}/{len(sp)} ({100*agree//len(sp)}%) '
          f'-- low = OSC +depth vs GT -depth convention, NOT an error')

region  corrected 3-axis
  ACB   23/23  (100%)
  CA1   27/29  (93%)
  CP    32/32  (100%)

DV sign agreement: 60/84 (71%) -- low = OSC +depth vs GT -depth convention, NOT an error


In [12]:
# Cell 8 — coverage / recall
gt_inj = {p for (p,_) in gt if p not in KNOWN_CANNULA}
matched = {r['pmid'] for r in results}
print(f'GT injection PMIDs     : {len(gt_inj)}')
print(f'  matched to OSC       : {len(matched & gt_inj)} '
      f'({100*len(matched & gt_inj)//max(len(gt_inj),1)}%)')
print(f'  PMID absent from OSC : {len(missing)}  {sorted({p for p,_ in missing})[:8]}')
print(f'  region not in OSC    : {len(no_region)}  {sorted({p for p,_ in no_region})[:8]}')

GT injection PMIDs     : 64
  matched to OSC       : 59 (92%)
  PMID absent from OSC : 3  ['19303923', '28486943']
  region not in OSC    : 4  ['27924869', '29184069', '30060138', '31447652']


In [13]:
# Cell 9 — residual taxonomy (Hungarian-honest: these are REAL, not mispairs)
misses = [r for r in full if not row_ok(r['gt'], r['osc'])]
def classify(r):
    g, o = r['gt'], r['osc']
    ea = best_abs_err(g['ap'], o['ap_raw'], ap=True)
    em = best_abs_err(g['ml'], o['ml_raw'])
    ed = best_abs_err(g['dv'], o['dv_raw'])
    if r['pmid'] in KNOWN_CANNULA: return 'cannula_offset'
    worst = max((e or 0, k) for e, k in ((ea,'AP'),(em,'ML'),(ed,'DV')))
    return f'{worst[1]}_off_{worst[0]:.1f}mm'
buckets = defaultdict(list)
for r in misses: buckets[classify(r)].append(r)
print(f'{len(misses)} residual misses (Hungarian-optimal — genuine, not pairing artifacts):')
for b in sorted(buckets, key=lambda k: -len(buckets[k])):
    print(f'\n  [{b}] x{len(buckets[b])}')
    for r in buckets[b]:
        g, o = r['gt'], r['osc']
        print(f'    {r["pmid"]} {r["region"]:4} '
              f'GT(ap{g["ap"]},ml{g["ml"]},dv{g["dv"]}) '
              f'OSC(ap{o["ap_raw"]},ml{o["ml_raw"]},dv{o["dv_raw"]})')

2 residual misses (Hungarian-optimal — genuine, not pairing artifacts):

  [AP_off_0.5mm] x1
    25043327 CA1  GT(ap-2.5,ml1.6,dv1.75) OSC(ap-2,ml-1.6,dv-1.7)

  [DV_off_0.6mm] x1
    32041202 CA1  GT(ap-2.0,ml1.3,dv1.3) OSC(ap-2,ml1.3,dv-1.9)


In [14]:
# Cell 10 — write audit CSV
import csv as _csv
out = []
for r in results:
    g, o = r['gt'], r['osc']
    out.append({'pmid':r['pmid'],'region':r['region'],
        'gt_ap':g['ap'],'gt_ml':g['ml'],'gt_dv':g['dv'],
        'osc_ap':o['ap_raw'],'osc_ml':o['ml_raw'],'osc_dv':o['dv_raw'],
        'ap_err':best_abs_err(g['ap'],o['ap_raw'],ap=True),
        'ml_err':best_abs_err(g['ml'],o['ml_raw']),
        'dv_err':best_abs_err(g['dv'],o['dv_raw']),
        'pass_3axis':'y' if row_ok(g,o) else 'n'})
with open('osc_injection_score_v31.csv','w',newline='',encoding='utf-8-sig') as f:
    w=_csv.DictWriter(f, fieldnames=list(out[0].keys())); w.writeheader(); w.writerows(out)
print('wrote osc_injection_score_v31.csv:', len(out), 'rows |',
      sum(r['pass_3axis']=='y' for r in out), 'pass')

wrote osc_injection_score_v31.csv: 84 rows | 82 pass
